# Price Prediction Regression Pipeline

This notebook documents the exploratory and modeling workflow for a tabular **price prediction** problem. It complements `regression_pipeline.py` by showing the reasoning behind the final pipeline in a more readable, notebook-style format.

## Goals
- Inspect the dataset and understand feature types
- Explore missingness, outliers, and the target distribution
- Build a leakage-safe regression workflow
- Compare several models using cross-validation
- Tune the strongest candidates
- Evaluate the final model with diagnostics and feature importance

## 1. Imports and configuration

Set the dataset path and target column below. The code uses **relative paths** so the notebook is portable inside the GitHub repository.

In [ ]:
from __future__ import annotations

import json
import math
import warnings
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.compose import ColumnTransformer, make_column_selector as selector
from sklearn.dummy import DummyRegressor
from sklearn.ensemble import ExtraTreesRegressor, HistGradientBoostingRegressor, RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.inspection import permutation_importance
from sklearn.linear_model import ElasticNet
from sklearn.metrics import mean_absolute_error, mean_squared_error, median_absolute_error, r2_score
from sklearn.model_selection import KFold, RandomizedSearchCV, cross_validate, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, RobustScaler

warnings.filterwarnings("ignore")

DATA_PATH = "test.csv"
TARGET_COL = "price_column"
OUTPUT_DIR = Path("regression_project_outputs_notebook")
OUTPUT_DIR.mkdir(exist_ok=True)

TEST_SIZE = 0.20
RANDOM_STATE = 42
CV_FOLDS = 5
HIGH_MISSING_THRESHOLD = 0.75
HIGH_ZERO_THRESHOLD = 0.75
MAX_SEARCH_ITERS = 20

## 2. Helper functions

These helpers keep the notebook tidy while still making the workflow readable.

In [ ]:
def sanitize_column_names(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    cleaned = []
    seen = {}

    for col in df.columns:
        new_col = str(col).strip().replace("\n", " ").replace("\t", " ")
        new_col = "_".join(new_col.split())

        if new_col in seen:
            seen[new_col] += 1
            new_col = f"{new_col}_{seen[new_col]}"
        else:
            seen[new_col] = 0

        cleaned.append(new_col)

    df.columns = cleaned
    return df


def load_table(path: str) -> pd.DataFrame:
    path_obj = Path(path)
    suffix = path_obj.suffix.lower()

    if suffix == ".csv":
        return pd.read_csv(path_obj)
    if suffix in {".xlsx", ".xls"}:
        return pd.read_excel(path_obj)
    if suffix == ".parquet":
        return pd.read_parquet(path_obj)

    raise ValueError(f"Unsupported file type: {suffix}")


def make_one_hot_encoder() -> OneHotEncoder:
    try:
        return OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    except TypeError:
        return OneHotEncoder(handle_unknown="ignore", sparse=False)


def regression_metrics(y_true: np.ndarray, y_pred: np.ndarray) -> dict:
    rmse = float(np.sqrt(mean_squared_error(y_true, y_pred)))
    denom = np.where(np.abs(y_true) < 1e-8, np.nan, np.abs(y_true))
    mape = float(np.nanmean(np.abs((y_true - y_pred) / denom)) * 100)

    return {
        "MAE": float(mean_absolute_error(y_true, y_pred)),
        "RMSE": rmse,
        "R2": float(r2_score(y_true, y_pred)),
        "MedianAE": float(median_absolute_error(y_true, y_pred)),
        "MAPE_percent": mape,
    }

## 3. Load and inspect the dataset

This first pass checks shape, schema, and whether the target column exists after basic name cleaning.

In [ ]:
df = load_table(DATA_PATH)
df = sanitize_column_names(df)
df = df.drop_duplicates().reset_index(drop=True)

print("Shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

if TARGET_COL not in df.columns:
    raise ValueError(
        f"TARGET_COL='{TARGET_COL}' not found.\nAvailable columns:\n{df.columns.tolist()}"
    )

display(df.head())
df.info()

## 4. Basic data quality checks

We inspect missingness, feature types, and the target distribution before modeling.

In [ ]:
missing_summary = (
    pd.DataFrame({
        "missing_count": df.isna().sum(),
        "missing_pct": df.isna().mean() * 100,
        "dtype": df.dtypes.astype(str)
    })
    .sort_values("missing_pct", ascending=False)
)

display(missing_summary.head(20))

numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
categorical_cols = df.select_dtypes(exclude=[np.number]).columns.tolist()

print("Numeric columns:", len(numeric_cols))
print("Categorical columns:", len(categorical_cols))
print("Duplicate rows:", df.duplicated().sum())

In [ ]:
plt.figure(figsize=(8, 5))
plt.hist(pd.to_numeric(df[TARGET_COL], errors="coerce").dropna(), bins=30)
plt.title("Target Distribution")
plt.xlabel(TARGET_COL)
plt.ylabel("Frequency")
plt.tight_layout()
plt.show()

## 5. Quick exploratory analysis

These plots help identify missingness patterns and the numeric features most related to the target.

In [ ]:
top_missing = (df.isna().mean() * 100).sort_values(ascending=False).head(20)

plt.figure(figsize=(10, 6))
plt.barh(top_missing.index[::-1], top_missing.values[::-1])
plt.title("Top 20 Columns by Missing Percentage")
plt.xlabel("Missing %")
plt.tight_layout()
plt.show()

In [ ]:
target_numeric = pd.to_numeric(df[TARGET_COL], errors="coerce")
corr_df = df.select_dtypes(include=[np.number]).copy()
corr_df[TARGET_COL] = target_numeric

correlations = (
    corr_df.corr(numeric_only=True)[TARGET_COL]
    .drop(TARGET_COL)
    .sort_values(key=lambda s: s.abs(), ascending=False)
)

display(correlations.head(15).to_frame("correlation"))

top_corr = correlations.head(15).sort_values()

plt.figure(figsize=(10, 6))
plt.barh(top_corr.index, top_corr.values)
plt.title("Top Numeric Correlations with Target")
plt.xlabel("Correlation")
plt.tight_layout()
plt.show()

## 6. Train/test split

The split happens **before** model fitting so the preprocessing and model selection process does not leak information from the test set.

In [ ]:
y = pd.to_numeric(df[TARGET_COL], errors="coerce")
X = df.drop(columns=[TARGET_COL]).copy()

valid_rows = y.notna()
X = X.loc[valid_rows].reset_index(drop=True)
y = y.loc[valid_rows].reset_index(drop=True)

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
)

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)

## 7. Preprocessing components

The notebook uses the same modeling logic as the main script:
- drop columns with very high missingness
- drop numeric columns dominated by zeros
- drop constant columns
- preprocess numeric and categorical variables separately

In [ ]:
class DropHighMissingColumns(BaseEstimator, TransformerMixin):
    def __init__(self, threshold: float = 0.75):
        self.threshold = threshold
        self.columns_to_drop_ = []

    def fit(self, X, y=None):
        X = pd.DataFrame(X).copy()
        missing_ratio = X.isna().mean()
        self.columns_to_drop_ = missing_ratio[missing_ratio > self.threshold].index.tolist()
        return self

    def transform(self, X):
        X = pd.DataFrame(X).copy()
        return X.drop(columns=self.columns_to_drop_, errors="ignore")


class DropHighZeroColumns(BaseEstimator, TransformerMixin):
    def __init__(self, threshold: float = 0.75):
        self.threshold = threshold
        self.columns_to_drop_ = []

    def fit(self, X, y=None):
        X = pd.DataFrame(X).copy()
        numeric_cols = X.select_dtypes(include=[np.number]).columns.tolist()

        if not numeric_cols:
            self.columns_to_drop_ = []
            return self

        zero_ratio = (X[numeric_cols] == 0).mean()
        self.columns_to_drop_ = zero_ratio[zero_ratio > self.threshold].index.tolist()
        return self

    def transform(self, X):
        X = pd.DataFrame(X).copy()
        return X.drop(columns=self.columns_to_drop_, errors="ignore")


class DropConstantColumns(BaseEstimator, TransformerMixin):
    def __init__(self):
        self.columns_to_drop_ = []

    def fit(self, X, y=None):
        X = pd.DataFrame(X).copy()
        nunique = X.nunique(dropna=False)
        self.columns_to_drop_ = nunique[nunique <= 1].index.tolist()
        return self

    def transform(self, X):
        X = pd.DataFrame(X).copy()
        return X.drop(columns=self.columns_to_drop_, errors="ignore")


def build_preprocessor() -> ColumnTransformer:
    numeric_pipeline = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", RobustScaler()),
    ])

    categorical_pipeline = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", make_one_hot_encoder()),
    ])

    return ColumnTransformer(
        transformers=[
            ("num", numeric_pipeline, selector(dtype_include=np.number)),
            ("cat", categorical_pipeline, selector(dtype_exclude=np.number)),
        ],
        remainder="drop",
    )


def build_full_pipeline(model) -> Pipeline:
    feature_cleaner = Pipeline(steps=[
        ("drop_high_missing", DropHighMissingColumns(threshold=HIGH_MISSING_THRESHOLD)),
        ("drop_high_zero", DropHighZeroColumns(threshold=HIGH_ZERO_THRESHOLD)),
        ("drop_constant", DropConstantColumns()),
    ])

    return Pipeline(steps=[
        ("feature_cleaner", feature_cleaner),
        ("preprocessor", build_preprocessor()),
        ("model", model),
    ])

## 8. Baseline model comparison

We compare several models using cross-validated RMSE, MAE, and R² on the training set.

In [ ]:
candidate_models = {
    "baseline_dummy": DummyRegressor(strategy="median"),
    "elastic_net": ElasticNet(max_iter=20000),
    "random_forest": RandomForestRegressor(
        n_estimators=300,
        random_state=RANDOM_STATE,
        n_jobs=-1,
    ),
    "extra_trees": ExtraTreesRegressor(
        n_estimators=400,
        random_state=RANDOM_STATE,
        n_jobs=-1,
    ),
    "hist_gradient_boosting": HistGradientBoostingRegressor(
        random_state=RANDOM_STATE
    ),
}

cv = KFold(n_splits=CV_FOLDS, shuffle=True, random_state=RANDOM_STATE)

rows = []
for name, model in candidate_models.items():
    pipeline = build_full_pipeline(model)

    scores = cross_validate(
        pipeline,
        X_train,
        y_train,
        cv=cv,
        scoring={
            "rmse": "neg_root_mean_squared_error",
            "mae": "neg_mean_absolute_error",
            "r2": "r2",
        },
        n_jobs=-1,
        error_score="raise",
        return_train_score=False,
    )

    rows.append({
        "model": name,
        "cv_rmse_mean": -scores["test_rmse"].mean(),
        "cv_rmse_std": scores["test_rmse"].std(),
        "cv_mae_mean": -scores["test_mae"].mean(),
        "cv_r2_mean": scores["test_r2"].mean(),
    })

leaderboard = pd.DataFrame(rows).sort_values("cv_rmse_mean")
display(leaderboard)

In [ ]:
plt.figure(figsize=(10, 5))
plt.bar(leaderboard["model"], leaderboard["cv_rmse_mean"])
plt.title("Cross-Validated RMSE by Model")
plt.ylabel("RMSE (lower is better)")
plt.xticks(rotation=25, ha="right")
plt.tight_layout()
plt.show()

## 9. Hyperparameter tuning

The top-performing non-baseline models are tuned with randomized search.

In [ ]:
param_grids = {
    "elastic_net": {
        "model__alpha": [0.0005, 0.001, 0.01, 0.1, 1.0, 10.0],
        "model__l1_ratio": [0.05, 0.2, 0.5, 0.8, 0.95, 1.0],
    },
    "random_forest": {
        "model__n_estimators": [300, 500, 700],
        "model__max_depth": [None, 8, 15, 25, 40],
        "model__min_samples_split": [2, 5, 10, 20],
        "model__min_samples_leaf": [1, 2, 4, 8],
        "model__max_features": [1.0, "sqrt", 0.5],
    },
    "extra_trees": {
        "model__n_estimators": [300, 500, 700],
        "model__max_depth": [None, 8, 15, 25, 40],
        "model__min_samples_split": [2, 5, 10, 20],
        "model__min_samples_leaf": [1, 2, 4, 8],
        "model__max_features": [1.0, "sqrt", 0.5],
    },
    "hist_gradient_boosting": {
        "model__learning_rate": [0.01, 0.03, 0.05, 0.1],
        "model__max_iter": [200, 400, 700],
        "model__max_leaf_nodes": [15, 31, 63, 127],
        "model__min_samples_leaf": [10, 20, 30, 50],
        "model__l2_regularization": [0.0, 0.01, 0.1, 1.0],
    },
}

top_candidates = [m for m in leaderboard["model"].tolist() if m != "baseline_dummy" and m in param_grids][:3]

best_name = None
best_model = None
best_score = math.inf
tuning_rows = []

for name in top_candidates:
    search = RandomizedSearchCV(
        estimator=build_full_pipeline(candidate_models[name]),
        param_distributions=param_grids[name],
        n_iter=min(MAX_SEARCH_ITERS, np.prod([len(v) for v in param_grids[name].values()])),
        scoring="neg_root_mean_squared_error",
        cv=cv,
        random_state=RANDOM_STATE,
        n_jobs=-1,
        verbose=1,
        refit=True,
    )
    search.fit(X_train, y_train)

    rmse = -search.best_score_
    tuning_rows.append({
        "model": name,
        "best_cv_rmse": rmse,
        "best_params": search.best_params_,
    })

    if rmse < best_score:
        best_score = rmse
        best_name = name
        best_model = search.best_estimator_

tuned_results = pd.DataFrame(tuning_rows).sort_values("best_cv_rmse")
display(tuned_results)

print("Best tuned model:", best_name)

## 10. Final evaluation on the held-out test set

In [ ]:
train_pred = best_model.predict(X_train)
test_pred = best_model.predict(X_test)

train_metrics = regression_metrics(y_train, train_pred)
test_metrics = regression_metrics(y_test, test_pred)

print("Train metrics:")
print(json.dumps(train_metrics, indent=2))

print("\nTest metrics:")
print(json.dumps(test_metrics, indent=2))

## 11. Diagnostic plots

These plots help evaluate calibration, error spread, and residual behavior.

In [ ]:
# Actual vs Predicted
plt.figure(figsize=(7, 6))
plt.scatter(y_test, test_pred, alpha=0.7)
min_val = min(np.min(y_test), np.min(test_pred))
max_val = max(np.max(y_test), np.max(test_pred))
plt.plot([min_val, max_val], [min_val, max_val], linewidth=2)
plt.xlabel("Actual")
plt.ylabel("Predicted")
plt.title("Actual vs Predicted")
plt.tight_layout()
plt.show()

# Residuals vs Predicted
residuals = y_test.values - test_pred
plt.figure(figsize=(7, 6))
plt.scatter(test_pred, residuals, alpha=0.7)
plt.axhline(0, linewidth=2)
plt.xlabel("Predicted")
plt.ylabel("Residual")
plt.title("Residuals vs Predicted")
plt.tight_layout()
plt.show()

# Residual distribution
plt.figure(figsize=(7, 6))
plt.hist(residuals, bins=30)
plt.xlabel("Residual")
plt.ylabel("Frequency")
plt.title("Residual Distribution")
plt.tight_layout()
plt.show()

## 12. Permutation feature importance

This gives a model-agnostic estimate of which original columns matter most for predictive performance.

In [ ]:
importance = permutation_importance(
    estimator=best_model,
    X=X_test,
    y=y_test,
    scoring="r2",
    n_repeats=10,
    random_state=RANDOM_STATE,
    n_jobs=-1,
)

importance_df = (
    pd.DataFrame({
        "feature": X_test.columns,
        "importance_mean": importance.importances_mean,
        "importance_std": importance.importances_std,
    })
    .sort_values("importance_mean", ascending=False)
)

display(importance_df.head(20))

top_features = importance_df.head(20).sort_values("importance_mean", ascending=True)

plt.figure(figsize=(10, 7))
plt.barh(top_features["feature"], top_features["importance_mean"], xerr=top_features["importance_std"])
plt.xlabel("Decrease in R² when Permuted")
plt.title("Top 20 Permutation Feature Importances")
plt.tight_layout()
plt.show()

## 13. Save the fitted notebook model (optional)

This saves the best fitted pipeline from the notebook run. The production-style artifact is still handled by `regression_pipeline.py`.

In [ ]:
joblib.dump(best_model, OUTPUT_DIR / "best_regression_pipeline_notebook.joblib")
print("Saved notebook model to:", OUTPUT_DIR / "best_regression_pipeline_notebook.joblib")

## 14. Conclusion

This notebook shows the reasoning behind the final project:
- inspect and clean a messy tabular dataset
- separate numeric and categorical preprocessing
- compare multiple regression models
- tune the strongest candidates
- evaluate the final model using both metrics and diagnostics

For a more reproducible, script-based version of the same project, see `regression_pipeline.py`.